# EMCRAG — preparazione dei 196 input
Aprire in **VS Code collegato a WSL Ubuntu-22.04**, selezionare il kernel **conda_ok_14apr** ed eseguire le celle in ordine.

Notebook autonomo: non richiede lo script `.py`. Verifica le sorgenti R0, associa i 96 record MMMED, scarica e valida le immagini, salva manifest e checksum. Non carica modelli e non modifica R0. I download validi sono riutilizzati. Le immagini storiche R0 non sono disponibili per un confronto dei byte.

In [ ]:
from pathlib import Path
import sys

R0_PATH = Path('/home/eugrug/Project_CRAG_MedGemma/ragas_results')
MMMED_PATH = Path('/mnt/f/processed_datasets/MMMED/Italian')
OUTPUT_PATH = Path('/home/eugrug/Project_CRAG_MedGemma/benchmark_m0_m1/inputs_196')

print('Python:', sys.executable)
for path in (R0_PATH, MMMED_PATH):
    if not path.is_dir():
        raise FileNotFoundError(f'Percorso non disponibile: {path}. Verificare il kernel WSL.')
print('Percorsi disponibili.')


## Funzioni
Eseguire questa cella una volta per definire le funzioni; non avvia download.

In [ ]:
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed
import hashlib
import io
import json
from pathlib import Path
from types import SimpleNamespace
import time
from urllib.request import Request, urlopen

RUN = 'ragas_v0611_20260719T010434Z'
HASHES = {
    'crag_outputs_selected_records.jsonl': '6dc949c69cb98a43d528af1d4bfd7a8bc5e5a5ce5d2ccabb990a1ec908969fa8',
    'ragas_extended_run_manifest.json': '70ee536db6136bf87d8a3ecbef35f0dd5b9f3573d80ae175698b7f8944682a37',
}

def sha(data):
    return hashlib.sha256(data).hexdigest()

def require(condition, message):
    if not condition:
        raise ValueError(message)

def encoded(obj):
    return (json.dumps(obj, ensure_ascii=False, sort_keys=True, indent=2) + '\n').encode()

def save_once(path, data):
    if path.exists():
        require(path.read_bytes() == data, f'File esistente diverso: {path}; nessuna sovrascrittura.')
    else:
        with path.open('xb') as f:
            f.write(data)

def inspect_image(data):
    from PIL import Image
    with Image.open(io.BytesIO(data)) as im:
        fmt, size = im.format, im.size
        im.verify()
    with Image.open(io.BytesIO(data)) as im:
        im.load()
    require(size[0] > 0 and size[1] > 0, 'Dimensioni immagine non valide')
    return {'image_sha256': sha(data), 'image_format': fmt,
            'image_width': size[0], 'image_height': size[1]}

def get_image(record, out):
    sid = record['sample_id']
    path = out / 'images' / (sid + '.image')
    receipt = path.with_suffix('.json')
    if path.exists():
        data = path.read_bytes()
    else:
        for attempt in range(3):
            try:
                req = Request(record['image_url'], headers={'User-Agent': 'EMCRAG-research/1.0'})
                with urlopen(req, timeout=40) as response:
                    data = response.read(25 * 1024 * 1024 + 1)
                require(len(data) <= 25 * 1024 * 1024, 'Immagine oltre 25 MiB')
                inspect_image(data)
                save_once(path, data)
                break
            except Exception:
                if attempt == 2:
                    raise
                time.sleep(attempt + 1)
    info = inspect_image(data)
    save_once(receipt, encoded({'source_url': record['image_url'], **info}))
    return {'image_path': path.relative_to(out).as_posix(), **info}

def prepare_inputs(r0_path, dataset_path, out_path):
    args = SimpleNamespace(r0=Path(r0_path), dataset=Path(dataset_path), out=Path(out_path))
    r0 = args.r0
    if not (r0 / 'crag_outputs_selected_records.jsonl').is_file():
        r0 = r0 / RUN
    raw = {}
    for name, expected in HASHES.items():
        raw[name] = (r0 / name).read_bytes()
        require(sha(raw[name]) == expected, f'Checksum R0 diverso: {name}. Stop per verificare la sorgente.')
    rows = [json.loads(s) for s in raw['crag_outputs_selected_records.jsonl'].decode().splitlines() if s.strip()]
    manifest = json.loads(raw['ragas_extended_run_manifest.json'])
    ids = [r['sample_id'] for r in rows]
    require(len(rows) == len(set(ids)) == 196, 'Conteggio/duplicati non validi')
    require(ids == manifest['dataset_configuration']['sample_ids'], 'Ordine diverso dal manifest R0')
    require(Counter(r['dataset'] for r in rows) == {'IMB_Curated_Test_Set_1000': 100, 'MMMED_Italian': 96}, 'Dataset inattesi')
    require(Counter(r['execution_mode'] for r in rows) == {'text_only': 100, 'multimodal': 96}, 'Modalita inattese')
    from datasets import load_from_disk
    ds = load_from_disk(str(args.dataset))
    require(len(ds) == 194, 'Dimensione MMMED diversa da 194')
    prepared = []
    for order, r in enumerate(rows):
        meta = r['metadata']
        mm = r['dataset'] == 'MMMED_Italian'
        item = {k: r[k] for k in ('sample_id', 'dataset', 'execution_mode', 'question', 'reference_answer')}
        item.update(benchmark_order=order, source_index=meta.get('source_index'),
                    sampling_seed=meta.get('sampling_seed'), options=meta.get('options'),
                    correct_letter=meta.get('correct_letter'), image_required=mm,
                    image_url=None, image_path=None, image_sha256=None)
        if mm:
            index = meta['source_index']
            require(isinstance(index, int) and 0 <= index < len(ds), f'Indice non valido: {r["sample_id"]}')
            source = ds[index]
            for field, archived in [('Question', r['question']), ('Correct Answer', r['reference_answer']), ('Correct Letter', meta['correct_letter'])]:
                require(str(source[field]).strip() == archived, f'{r["sample_id"]}: differenza nel campo {field}')
            for letter, value in meta['options'].items():
                require(str(source['Answer' + letter]).strip() == value, f'{r["sample_id"]}: opzione {letter} diversa')
            url = source['Image url']
            require(isinstance(url, str) and url.startswith(('https://', 'http://')), f'URL non valido: {r["sample_id"]}')
            item['image_url'] = url
        prepared.append(item)
    print('PASS: sorgenti R0, 196 ID, ordine, 100 IMB + 96 MMMED, corrispondenza MMMED.', flush=True)
    (args.out / 'images').mkdir(parents=True, exist_ok=True)
    errors = []
    with ThreadPoolExecutor(max_workers=4) as pool:
        pending = {pool.submit(get_image, r, args.out): r for r in prepared if r['image_required']}
        for completed, future in enumerate(as_completed(pending), 1):
            r = pending[future]
            try:
                r.update(future.result())
                print(f'[{completed}/96] OK {r["sample_id"]}', flush=True)
            except Exception as exc:
                errors.append((r['sample_id'], str(exc)))
                print(f'[{completed}/96] ERRORE {r["sample_id"]}: {exc}', flush=True)
    if errors:
        print(f'STOP: {len(errors)} immagini non disponibili. Nessun manifest finale creato. I download validi sono conservati.')
        return 1
    data = ''.join(json.dumps(r, ensure_ascii=False, sort_keys=True) + '\n' for r in prepared).encode()
    name = 'EMCRAG_manifest_196.jsonl'
    audit = {'status': 'PASS', 'records': 196, 'IMB': 100, 'MMMED': 96,
             'images_validated': 96, 'r0_order_verified': True,
             'mmmed_source_fields_match_after_R0_strip': True, 'source_sha256': HASHES,
             'manifest_sha256': sha(data),
             'image_identity_vs_historical_R0': 'Not verifiable: historical image bytes/hashes absent from R0 archive.',
             'image_policy': 'Current downloaded bytes frozen for identical M0/M1 inputs; no resampling.'}
    save_once(args.out / name, data)
    save_once(args.out / 'SHA256SUMS.txt', (sha(data) + '  ' + name + '\n').encode())
    save_once(args.out / 'input_audit.json', encoded(audit))
    print('\nPASS: 196 record; 96 immagini valide; manifest e checksum salvati.')
    print('Output:', args.out)
    print('Manifest SHA256:', sha(data))
    return 0


## Esecuzione
Questa cella avvia i controlli e il download. Attendere il riepilogo **PASS**. In caso di errore, conservare l'output per la diagnosi; non modificare i casi del benchmark.

In [ ]:
status = prepare_inputs(R0_PATH, MMMED_PATH, OUTPUT_PATH)
if status != 0:
    raise RuntimeError('Preparazione incompleta: vedere gli errori sopra. I download validi sono conservati.')
